# T4 re-evaluation — per-optimization attribution

Runs `docs/t4-reevaluation-plan.md` on one Colab T4 (Kaggle T4x2 also works; the queue runs on GPU 0 only,
because every engine A/B is host-sensitive and must not share a card). Three sessions, each self-contained
with its own baselines; results checkpoint to `$RUN/` after every job, so a dead session resumes from the
first job whose JSON is missing.

Colab: Runtime → Change runtime type → **T4 GPU**. Mount Drive in the next cell if you want `$RUN` to
survive the session; otherwise download the zip at the end.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.used,clocks.max.sm,power.limit --format=csv
import os, subprocess
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
GPUS = [0]   # one card only: engine A/Bs are host-sensitive and interleaving handles drift
print("kaggle" if ON_KAGGLE else "colab", "| using GPU 0")

In [ ]:
# Optional on Colab: keep results across sessions.
MOUNT_DRIVE = False
if MOUNT_DRIVE and not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
os.chdir("/kaggle/working" if ON_KAGGLE else "/content")
if not os.path.isdir("fol"):
    !git clone -b t4-phase0 https://github.com/Vaibhav7711/full-inference-engine.git fol
%cd fol
!git pull -q && git log -1 --oneline
%env TOKENIZERS_PARALLELISM=false
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
!bash scripts/{'setup_kaggle' if ON_KAGGLE else 'setup_colab'}.sh


In [ ]:
import datetime, pathlib
SHA = subprocess.check_output(["git","rev-parse","--short","HEAD"], text=True).strip()
RUN = (f"/content/drive/MyDrive/t4_reeval/{datetime.date.today():%Y%m%d}_{SHA}"
       if (MOUNT_DRIVE and not ON_KAGGLE) else f"results/t4/{datetime.date.today():%Y%m%d}_{SHA}")
pathlib.Path(RUN).mkdir(parents=True, exist_ok=True)
%env RUN=$RUN
!python scripts/colab_preflight.py --out $RUN/env.json
print(RUN)

## Runner

Each job is its own process on GPU 0, logged to `$RUN/logs/<name>.log`. `done()` skips jobs whose output already exists, so re-running a session cell after a crash resumes.

In [ ]:
import subprocess, threading, time, os, pathlib

def gpu_used_mb(index=0):
    return int(subprocess.check_output(["nvidia-smi", f"--id={index}", "--query-gpu=memory.used",
                                        "--format=csv,noheader,nounits"], text=True))

def gpu_holders():
    """Every process that could be holding GPU memory: nvidia-smi's view, then any other python."""
    me = os.getpid()
    try:
        apps = subprocess.check_output(["nvidia-smi", "--query-compute-apps=pid,used_memory,process_name",
                                        "--format=csv,noheader"], text=True).strip()
    except subprocess.CalledProcessError:
        apps = ""
    ps = subprocess.check_output(["ps", "-eo", "pid,ppid,etime,rss,cmd"], text=True).splitlines()
    pythons = [line for line in ps[1:] if "python" in line and f" {me} " not in line.split("python")[0]
               and int(line.split()[0]) != me]
    return apps, pythons

def kill_stray_python(dry_run=True):
    """Kill every python process except this kernel (and its parent). Run once with dry_run=True first."""
    me, parent = os.getpid(), os.getppid()
    _, pythons = gpu_holders()
    for line in pythons:
        pid = int(line.split()[0])
        if pid in (me, parent):
            continue
        print(("would kill " if dry_run else "killing ") + line.strip()[:160])
        if not dry_run:
            subprocess.call(["kill", "-9", str(pid)])
    if not dry_run:
        time.sleep(2); print({g: f"{gpu_used_mb(g)} MiB" for g in GPUS})

def assert_gpus_free(limit_mb=600):
    for g in GPUS:
        used = gpu_used_mb(g)
        if used >= limit_mb:
            apps, pythons = gpu_holders()
            raise AssertionError(
                f"GPU {g}: {used} MiB in use.\n"
                f"nvidia-smi compute apps:\n{apps or '  (none visible from inside the container)'}\n"
                f"other python processes (interrupted run_queues jobs keep running; VS Code terminals count):\n  "
                + "\n  ".join(line.strip()[:160] for line in pythons)
                + "\nRun kill_stray_python(dry_run=False) to clear them, or stop the process in its terminal.")
    print("GPUs free:", {g: f"{gpu_used_mb(g)} MiB" for g in GPUS})

def run_queues(queues, tail=8):
    """queues: list of [(name, command), ...]; queue i runs on GPU i (all on GPU 0 if only one)."""
    logs = pathlib.Path(RUN) / "logs"; logs.mkdir(exist_ok=True)
    if len(GPUS) == 1:
        queues = [[job for queue in queues for job in queue]]
    results = {}
    def worker(gpu, queue):
        for name, command in queue:
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
            started = time.perf_counter()
            with open(logs / f"{name}.log", "w") as log:
                code = subprocess.call(command, shell=True, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=os.getcwd())
            results[name] = (gpu, code, time.perf_counter() - started)
            print(f"[gpu{gpu}] {name}: exit {code} in {results[name][2]/60:.1f} min", flush=True)
    threads = [threading.Thread(target=worker, args=(GPUS[i], queue)) for i, queue in enumerate(queues)]
    for t in threads: t.start()
    for t in threads: t.join()
    for name, (gpu, code, seconds) in results.items():
        print(f"\n===== {name} (gpu{gpu}, exit {code}) =====")
        print("".join(open(logs / f"{name}.log").readlines()[-tail:]))
    failed = [name for name, (_, code, _) in results.items() if code]
    assert not failed, f"failed: {failed} - see {logs}"

In [ ]:
def done(name_cmd):
    """Drop jobs whose output JSON already exists, so a re-run resumes rather than repeats."""
    import re
    kept = []
    for name, cmd in name_cmd:
        m = re.search(r"--out(?:put)?\s+(\S+)", cmd)
        if m and os.path.exists(os.path.expandvars(m.group(1))):
            print(f"skip {name}: {m.group(1)} exists")
        else:
            kept.append((name, cmd))
    return kept

## Gate — correctness before any timing

The fusion toggles and leave-one-out arms are new; a `False` arm that changes tokens is a bug, and the A/B
would refuse to run anyway. Kernel/cache/server suites, then the continuous-batching CUDA suite (includes the
new `test_d3_each_fusion_toggle_off_matches_ref`).

In [ ]:
assert_gpus_free()
PYTEST = "python -m pytest -q -m cuda -p no:cacheprovider"
run_queues([[
    ("gate_kernels_cache_server", f"{PYTEST} tests/kernels tests/cache tests/correctness tests/server"),
    ("gate_batching", f"{PYTEST} tests/batching --ignore=tests/batching/test_batched_speculative.py"),
]], tail=4)

## Session A — baselines and toggles (~75 min)

B0 stock HF, B2 roofline, then every runtime-toggleable optimization leave-one-out against `full`
(graphs + prefix cache + tiled prefill + all three fusions). `loo_all` interleaves all seven arms in one run so
they share one `full` baseline and one thermal history; the fusion pairs are then re-run with graphs **off**,
because their launch-cost savings overlap with what graphs remove.

In [ ]:
assert_gpus_free()
AB = "python -m benchmarks.reliability.ab --repeats 5 --prompt-profile chat"
session_a = [
    ("A2_roofline",        "python -m benchmarks.kernels.roofline --out $RUN/roofline.json"),
    ("A3_b0_stock",        "python -m benchmarks.inference.t4_baseline --include-bfloat16 --output $RUN/b0_stock_baseline.json"),
    ("A4_paged_cache",     "python -m benchmarks.cache.paged_cache_bench --block-sizes 8,16,32 --output $RUN/paged_cache_bench.json"),
    # leave-one-out, graphs present (the serving configuration)
    ("A5_loo_all",         f"{AB} --setting loo_all --out $RUN/ab_loo_all_chat.json"),
    # Phase 7 vs 8 and Phase 9 re-run
    ("A6_buckets_padded",  f"{AB} --setting graph_buckets_padded --out $RUN/ab_graph_buckets_padded_chat.json"),
    ("A7_mlp_gate_up",     f"{AB} --setting mlp_gate_up --cuda-graphs --out $RUN/ab_mlp_gate_up_chat.json"),
    # fusions with graphs OFF: their value before graphs existed
    ("A8_rmsnorm_nograph", f"{AB} --setting triton_rmsnorm --out $RUN/ab_triton_rmsnorm_nograph_chat.json"),
    ("A9_rope_nograph",    f"{AB} --setting triton_rope    --out $RUN/ab_triton_rope_nograph_chat.json"),
    ("A10_swiglu_nograph", f"{AB} --setting triton_swiglu  --out $RUN/ab_triton_swiglu_nograph_chat.json"),
    # prefill policy and prefix cache, graphs on
    ("A11_prefill_chunk",  f"{AB} --setting prefill_chunk --cuda-graphs --out $RUN/ab_prefill_chunk_chat.json"),
    ("A12_prefix_ttft",    "python -m benchmarks.batching.prefix_cache_ttft --output $RUN/prefix_cache_ttft.json"),
    ("A13_warmup",         f"{AB} --setting warmup --cuda-graphs --out $RUN/ab_warmup_chat.json"),
    ("A14_token_transfer", "python -m benchmarks.batching.token_transfer_ab --widths 1,2,4,8,16 --output $RUN/token_transfer_ab.json"),
]
run_queues([done(session_a)], tail=6)

## Session B — commit ladder (~60–70 min)

Structural changes with no toggle (D3–D9, P1, R2). One git worktree per rung, every rung every round, widths
1/8/16 on the journal's 16×32 workload. Prints per-rung medians ± spread and each rung's verdict against the
previous one. Old rungs use their own `continuous_throughput.py`; the JSON schema is identical back to `79d8671`.

In [ ]:
assert_gpus_free()
run_queues([[("B_ladder", "python scripts/commit_ladder.py --rounds 3 --widths 1,8,16 --out $RUN/ladder.json")]], tail=22)

## Session C — long context, kernels, mixed arrival (~50 min)

In [ ]:
assert_gpus_free()
session_c = [
    ("C1_kv_dtype_long",   "python -m benchmarks.reliability.ab --repeats 5 --setting kv_dtype --prompt-profile long --cuda-graphs --num-blocks 512 --out $RUN/ab_kv_dtype_long.json"),
    ("C2_int8_kernel",     "python -m benchmarks.kernels.int8_paged_decode_ab --seq-lens 256,1024,2048 --batches 1,16 --rounds 5 --output $RUN/int8_paged_decode_ab.json"),
    ("C3_decode_regimes",  "python -m benchmarks.kernels.paged_decode_regime_sweep --seq-lens 64,128,256,512,1024,2048 --batches 1,8,16 --configs 64x4,128x4 --output $RUN/paged_decode_regime_sweep.json"),
    ("C4_prefill_sweep",   "python -m benchmarks.reliability.sweep --only B1 B3 --out $RUN/prefill_sweep.json"),
    ("C5_prefill_tiles",   "python -m benchmarks.kernels.prefill_attention_ab --sweep-tiles --chunks 64 128 256 512 --out $RUN/prefill_attention_sweep.json"),
    ("C6_prefill_budget",  "python -m benchmarks.batching.mixed_arrival_prefill_budget_ab --budgets 64,128,256 --rounds 5 --output $RUN/mixed_arrival_prefill_budget_ab.json"),
    ("C7_mixed_graphs",    "python -m benchmarks.batching.mixed_arrival_graph_ab --output $RUN/mixed_arrival_graph_ab.json"),
    ("C8_occupancy",       "python -m benchmarks.batching.padded_graph_occupancy_ab --output $RUN/padded_graph_occupancy_ab.json"),
    ("C9_w8a16",           "python -m benchmarks.kernels.w8a16_linear_ab --output $RUN/w8a16_linear_ab.json"),
    ("C10_w8a8",           "python -m benchmarks.kernels.w8a8_linear_ab --output $RUN/w8a8_linear_ab.json"),
    ("C11_uvicorn",        "python -m benchmarks.server.uvicorn_load --requests 16 --output $RUN/uvicorn_load.json"),
]
run_queues([done(session_c)], tail=6)

## Ledger

One block per A/B with every variant arm's verdict against its baseline, then the ladder table. `unresolved`
is recorded as `≤ spread`, never as a small win; any non-`kv_dtype` A/B with `tokens identical=False` is a
kernel bug, not a result. Paste this into `results/t4/<run>/ledger.md` next to the journal claims in
`docs/t4-reevaluation-plan.md` §3.

In [ ]:
import json, glob
LEDGER = ("step_timing.expected_gap_ms", "latency.itl_p50", "latency.itl_p99", "latency.ttft_p50",
          "step_timing.decode_step_p50_ms", "step_timing.prefill_step_p50_ms", "step_timing.host_stage_ms_p50")
for path in sorted(glob.glob(f"{RUN}/ab_*.json")):
    d = json.load(open(path)); labels = list(d["arms"]); base = labels[0]
    print(f"\n{os.path.basename(path)} baseline={base} tokens identical={d['token_identity']['identical']} "
          f"SM MHz before/after: {d['clocks_before'].get('clocks.sm')}/{d['clocks_after'].get('clocks.sm')}")
    for variant, comp in d.get("comparisons", {base: d["comparison"]}).items():
        print(f"  -- {variant} vs {base}")
        for m in LEDGER:
            b = d["arms"][base]["summary"].get(m, {})
            print(f"     {m:34s} base={b.get('median', float('nan')):8.3f}  {comp.get(m, '-')}")
if os.path.exists(f"{RUN}/ladder.json"):
    d = json.load(open(f"{RUN}/ladder.json")); w = d["widths"][-1]
    print(f"\nladder @ width {w}")
    for label in d["order"]:
        r = d["rungs"][label]; s = r.get("summary", {}).get(str(w)) or r.get("summary", {}).get(w, {})
        med = f"{s['median']:8.1f} ±{s['spread']:4.0%}" if s.get("n") else "     n/a"
        print(f"  {label:36s} {r['sha']:8s} {med}  {r.get('vs_previous', ''):30s} journal: {r['claim']}")

## Save before the session dies

In [ ]:
!git add $RUN 2>/dev/null; git -c user.name=t4-runner -c user.email=t4@local commit -q -m "T4 re-evaluation results $RUN" && git log -1 --oneline
!zip -qr ../t4_reeval_results.zip $RUN && ls -la ../t4_reeval_results.zip
# Colab: from google.colab import files; files.download("../t4_reeval_results.zip")